In [ ]:
import datetime
import os
from src.utils.config import *
from src.ingestion.aviation_edge import fetch_flight_history
from src.ingestion.loaders import load_json_to_spark_df
from src.utils.spark import get_spark

In [ ]:
spark = get_spark()

ch_user = os.getenv('CLICKHOUSE_USER')
ch_password = os.getenv('CLICKHOUSE_PW')
ch_host = os.getenv('CLICKHOUSE_HOST')
client = setup_clickhouse_client(ch_user, ch_password, ch_host)

api_key = "YOUR_API_KEY"  # replace later with os.getenv('AE_API_KEY')

airport_code = "SIN"
flight_types = ["arrival", "departure"]

start_date = datetime.datetime(2025, 9, 1)
end_date = datetime.datetime(2025, 9, 5)

column_names = ['type', 'status', 'departure', 'arrival',
  'airline', 'flight', 'codeshared_airline', 'codeshared_flight']
defaults = {
  'type': '',
  'status': '',
  'departure': {},
  'arrival': {},
  'airline': {},
  'flight': {},
  'codeshared_airline': {},
  'codeshared_flight': {}
}

In [ ]:
records = fetch_flight_history(
    api_key=api_key,
    airport_code=airport_code,
    flight_types=flight_types,
    start_date=start_date,
    end_date=end_date,
)

print(f"Total records: {len(records)}")

In [ ]:
df = load_json_to_spark_df(spark, records)

df.head()

In [ ]:
output_path = "/content/drive/MyDrive/aviation-delay/data/raw/flights.csv"

df.to_csv(output_path, index=False)

print(f"Saved to {output_path}")

In [ ]:
# kafka loading from csv
boostrap_servers = os.getenv('KAFKA_BOOTSTRAP_SERVERS')
hist_av_prod = create_kafka_producer('historical_aviation_data', boostrap_servers)
hist_av_cons = create_kafka_consumer('consume_aviation_data', bootstrap_servers, 'raw_aviation_data')

# give the data to kafka topic to process
stream_to_kafka(output_path, hist_av_prod, 'raw_aviation_data')
# insert data to clickhouse for storage
insert_to_clickhouse(hist_av_cons, 'raw_aviation_flights', client, column_names, defaults)

